# IMDB comparative sentiment experiment
Arnav Kumar · 230911308 · ICT-4442

Run with a GPU runtime. This notebook calls the same tested source files; it does not contain fabricated outputs. Locally, open it in the extracted project folder. In Colab, upload the supplied project ZIP in the next cell.

In [ ]:
from pathlib import Path
import os, sys, zipfile
if not Path('run_all.py').exists():
    from google.colab import files
    uploaded = files.upload()
    archive = next(name for name in uploaded if name.endswith('.zip'))
    target = Path('/content/imdb_project').resolve()
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            destination = (target / member.filename).resolve()
            if target not in destination.parents and destination != target:
                raise ValueError('Invalid archive path')
        z.extractall(target)
    os.chdir(next(target.rglob('run_all.py')).parent)
print(Path.cwd())


In [ ]:
%pip install -r requirements.txt


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU: training will be slow')


## Verify the code
Offline regression tests cover padding invariance, vocabulary isolation, gradient accumulation and saved-model prediction.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## Full experiment
The 50,000-review run uses two learning-rate candidates per model and three epochs per candidate. Selected checkpoints are evaluated once on test. On a clean run, downloads take additional time. To retrain supplied completed results, set `--output-root results_new`; canonical documents read `results/`.

In [ ]:
subprocess.run([sys.executable, "run_all.py", "--epochs", "3"], check=True)


In [ ]:
import pandas as pd
pd.read_csv("results/comparison.csv")


In [ ]:
from IPython.display import display, Image
for name in ["learning_curves", "confusion_matrices", "tradeoffs"]:
    display(Image(filename=f"results/{name}.png"))


## Predict a review
Saved bundles contain the exact fitted preprocessing.

In [ ]:
subprocess.run([sys.executable, "-m", "src.predict", "--model-dir", "results/mlp", "--text", "A beautifully acted and moving film."], check=True)


## Export
The archive below excludes downloaded data and duplicate tuning checkpoints. It includes selected trained bundles, results, report and slides.

In [ ]:
with zipfile.ZipFile('IMDB_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['results', 'docs']:
        for path in Path(folder).rglob('*'):
            if path.is_file() and '__pycache__' not in path.parts:
                if 'tuning' in path.parts and path.suffix in ['.pt', '.joblib']:
                    continue
                z.write(path)
try:
    from google.colab import files
    files.download('IMDB_results.zip')
except ImportError:
    print(Path('IMDB_results.zip').resolve())
